Étape 2 — Scoring pédagogique et calcul des niveaux
Attention : cette étape nécessite les réponses correctes / barèmes des tests.
Sans barème, le code peut préparer les colonnes et la structure, mais il ne peut pas calculer un score fiable.

Donc l’objectif de cette Étape 2 sont:
1. charger les fichiers préparés de l’Étape 1 ;
2. détecter les questions ;
3. créer les dictionnaires de correction ;
4. calculer les scores initiaux par domaine ;
5. calculer les scores intermédiaires ;
6. calculer les scores finaux ;
7. calculer les pourcentages ;
8. attribuer les niveaux : Débutant, Intermédiaire, Avancé ;
9. calculer la progression ;
10. exporter les datasets scorés.

Cellule 1 — Importation des bibliothèques

In [1]:
# ============================================================
# ÉTAPE 2 — SCORING PÉDAGOGIQUE ET CALCUL DES NIVEAUX
# Projet : Prédiction et évaluation de la performance académique
# ============================================================

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

Cellule 2 — Définition des chemins

In [2]:
# ============================================================
# 2.1. Chemins des fichiers préparés
# ============================================================

INPUT_DIR = Path("outputs/01_integration_preparation")
OUTPUT_DIR = Path("outputs/02_scoring_pedagogique")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_INSCRIPTIONS = INPUT_DIR / "01_inscriptions_preparees.csv"
PATH_TEST_INITIAL = INPUT_DIR / "02_test_initial_prepare.csv"
PATH_TESTS_INTERMEDIAIRES = INPUT_DIR / "03_tests_intermediaires_prepares.csv"
PATH_TESTS_FINAUX = INPUT_DIR / "04_tests_finaux_prepares.csv"

PATH_DATASET_ORIENTATION = INPUT_DIR / "05_dataset_orientation_initiale.csv"
PATH_DATASET_PREDICTION = INPUT_DIR / "06_dataset_prediction_finale_pre_scoring.csv"
PATH_DATASET_LONGITUDINAL = INPUT_DIR / "07_dataset_longitudinal_pre_scoring.csv"

fichiers = {
    "inscriptions": PATH_INSCRIPTIONS,
    "test_initial": PATH_TEST_INITIAL,
    "tests_intermediaires": PATH_TESTS_INTERMEDIAIRES,
    "tests_finaux": PATH_TESTS_FINAUX,
    "dataset_orientation": PATH_DATASET_ORIENTATION,
    "dataset_prediction": PATH_DATASET_PREDICTION,
    "dataset_longitudinal": PATH_DATASET_LONGITUDINAL
}

for nom, chemin in fichiers.items():
    print(f"{nom:25s} | existe = {chemin.exists()} | {chemin}")

inscriptions              | existe = True | outputs/01_integration_preparation/01_inscriptions_preparees.csv
test_initial              | existe = True | outputs/01_integration_preparation/02_test_initial_prepare.csv
tests_intermediaires      | existe = True | outputs/01_integration_preparation/03_tests_intermediaires_prepares.csv
tests_finaux              | existe = True | outputs/01_integration_preparation/04_tests_finaux_prepares.csv
dataset_orientation       | existe = True | outputs/01_integration_preparation/05_dataset_orientation_initiale.csv
dataset_prediction        | existe = True | outputs/01_integration_preparation/06_dataset_prediction_finale_pre_scoring.csv
dataset_longitudinal      | existe = True | outputs/01_integration_preparation/07_dataset_longitudinal_pre_scoring.csv


Cellule 3 — Chargement des datasets

In [3]:
# ============================================================
# 2.2. Chargement des données préparées
# ============================================================

df_inscriptions = pd.read_csv(PATH_INSCRIPTIONS)
df_test_initial = pd.read_csv(PATH_TEST_INITIAL)
df_tests_intermediaires = pd.read_csv(PATH_TESTS_INTERMEDIAIRES)
df_tests_finaux = pd.read_csv(PATH_TESTS_FINAUX)

df_orientation = pd.read_csv(PATH_DATASET_ORIENTATION)
df_prediction = pd.read_csv(PATH_DATASET_PREDICTION)
df_longitudinal = pd.read_csv(PATH_DATASET_LONGITUDINAL)

print("Chargement terminé.")
print("Inscriptions :", df_inscriptions.shape)
print("Test initial :", df_test_initial.shape)
print("Tests intermédiaires :", df_tests_intermediaires.shape)
print("Tests finaux :", df_tests_finaux.shape)
print("Dataset orientation :", df_orientation.shape)
print("Dataset prédiction :", df_prediction.shape)
print("Dataset longitudinal :", df_longitudinal.shape)

Chargement terminé.
Inscriptions : (472, 20)
Test initial : (305, 23)
Tests intermédiaires : (196, 20)
Tests finaux : (206, 23)
Dataset orientation : (305, 42)
Dataset prédiction : (128, 64)
Dataset longitudinal : (125, 84)


Cellule 4 — Fonctions générales de scoring

In [4]:
# ============================================================
# 2.3. Fonctions générales de scoring
# ============================================================

def normaliser_reponse(valeur):
    """
    Normalise une réponse pour faciliter la comparaison :
    - conversion en texte ;
    - suppression des espaces multiples ;
    - mise en minuscules ;
    - gestion des valeurs manquantes.
    """
    if pd.isna(valeur):
        return np.nan
    
    valeur = str(valeur).strip()
    valeur = re.sub(r"\s+", " ", valeur)
    
    if valeur.lower() in ["", "nan", "none", "nat", "null"]:
        return np.nan
    
    return valeur.lower()


def est_reponse_inconnue(valeur):
    """
    Détecte les réponses inconnues ou non renseignées.
    """
    if pd.isna(valeur):
        return True
    
    valeur_norm = normaliser_reponse(valeur)
    
    if pd.isna(valeur_norm):
        return True
    
    motifs_inconnus = [
        "je ne sais pas",
        "ne sais pas",
        "aucune réponse",
        "sans réponse",
        "non répondu",
        "non repondu",
        "inconnu",
        "unknown"
    ]
    
    return any(motif in valeur_norm for motif in motifs_inconnus)


def corriger_reponse(reponse, bonne_reponse):
    """
    Retourne :
    - 1 si la réponse est correcte ;
    - 0 si la réponse est incorrecte ;
    - 0 si la réponse est inconnue ou manquante.
    
    NB : dans cette version, une réponse manquante est considérée comme incorrecte.
    """
    if est_reponse_inconnue(reponse):
        return 0
    
    if bonne_reponse is None or pd.isna(bonne_reponse):
        return np.nan
    
    reponse_norm = normaliser_reponse(reponse)
    bonne_reponse_norm = normaliser_reponse(bonne_reponse)
    
    if pd.isna(reponse_norm) or pd.isna(bonne_reponse_norm):
        return 0
    
    return int(reponse_norm == bonne_reponse_norm)


def attribuer_niveau_depuis_pourcentage(score_pct):
    """
    Attribue un niveau de compétence selon le score en pourcentage.
    
    Règle officielle du mémoire :
    - Débutant : score < 50 %
    - Intermédiaire : 50 % <= score < 70 %
    - Avancé : score >= 70 %
    """
    if pd.isna(score_pct):
        return np.nan
    
    if score_pct < 50:
        return "Débutant"
    elif score_pct < 70:
        return "Intermédiaire"
    else:
        return "Avancé"


def afficher_shape(nom, df):
    print(f"{nom:45s} : {df.shape[0]} lignes | {df.shape[1]} colonnes")

Cellule 5— Détection numérique correcte Q1 à Q20

In [8]:
# ============================================================
# 2.4. Détection des colonnes de questions — VERSION CORRIGÉE
# ============================================================

def extraire_numero_question_initiale(colonne):
    """
    Extrait correctement le numéro d'une question initiale.
    Exemples :
    - Q1 - Resume ventes Excel  -> 1
    - Q10 - 12 commerciaux      -> 10
    - Q20 - Sortie fiable LLM   -> 20
    """
    col = str(colonne).strip()
    match = re.match(r"^Q\s*(\d{1,2})(\s|-|_|:|\.|$)", col, flags=re.IGNORECASE)
    
    if match:
        numero = int(match.group(1))
        if 1 <= numero <= 20:
            return numero
    
    return None


def detecter_questions_initiales(df):
    """
    Détecte et trie correctement les colonnes Q1 à Q20.
    """
    colonnes_questions = []
    
    for col in df.columns:
        numero = extraire_numero_question_initiale(col)
        if numero is not None:
            colonnes_questions.append((numero, col))
    
    colonnes_questions = sorted(colonnes_questions, key=lambda x: x[0])
    
    return [col for numero, col in colonnes_questions]


def detecter_questions_standardisees(df, prefixe):
    """
    Détecte les colonnes standardisées :
    mi_Q01 à mi_Q15
    final_Q01 à final_Q15
    """
    pattern = rf"^{prefixe}_Q(\d+)"
    
    colonnes_questions = []
    
    for col in df.columns:
        match = re.match(pattern, str(col), flags=re.IGNORECASE)
        if match:
            numero = int(match.group(1))
            colonnes_questions.append((numero, col))
    
    colonnes_questions = sorted(colonnes_questions, key=lambda x: x[0])
    
    return [col for numero, col in colonnes_questions]


colonnes_q_initial = detecter_questions_initiales(df_test_initial)
colonnes_q_mi = detecter_questions_standardisees(df_tests_intermediaires, "mi")
colonnes_q_final = detecter_questions_standardisees(df_tests_finaux, "final")

print("Questions initiales détectées :", len(colonnes_q_initial))
for i, col in enumerate(colonnes_q_initial, start=1):
    print(f"Q{i:02d} -> {col}")

print("\nQuestions intermédiaires détectées :", len(colonnes_q_mi))
print(colonnes_q_mi)

print("\nQuestions finales détectées :", len(colonnes_q_final))
print(colonnes_q_final)

Questions initiales détectées : 20
Q01 -> Q1 - Resume ventes Excel
Q02 -> Q2 - Import CSV Excel
Q03 -> Q3 - Modele donnees Excel
Q04 -> Q4 - Mauvaise pratique visu Excel
Q05 -> Q5 - 2e grande valeur Excel
Q06 -> Q6 - Mesure vs Colonne DAX
Q07 -> Q7 - CA annee precedente DAX
Q08 -> Q8 - Vue Modele Power BI
Q09 -> Q9 - Acces directeurs regionaux
Q10 -> Q10 - 12 commerciaux 3 indicateurs
Q11 -> Q11 - Bibliotheque CSV Python
Q12 -> Q12 - Overfitting Underfitting
Q13 -> Q13 - Segmentation 50000 clients
Q14 -> Q14 - Deployer modele Python API
Q15 -> Q15 - Valeurs manquantes 30pc
Q16 -> Q16 - Role system prompt LLM
Q17 -> Q17 - Assistant IA PDF financiers
Q18 -> Q18 - Role embedding dans RAG
Q19 -> Q19 - Agent IA selection outil
Q20 -> Q20 - Sortie fiable LLM tableau

Questions intermédiaires détectées : 15
['mi_Q01', 'mi_Q02', 'mi_Q03', 'mi_Q04', 'mi_Q05', 'mi_Q06', 'mi_Q07', 'mi_Q08', 'mi_Q09', 'mi_Q10', 'mi_Q11', 'mi_Q12', 'mi_Q13', 'mi_Q14', 'mi_Q15']

Questions finales détectées : 15
['f

Cellule 6  — Répartition correcte par domaine

In [9]:
# ============================================================
# 2.5. Répartition des questions initiales par domaine
# VERSION CORRIGÉE
# ============================================================

# Sécurité : vérifier que les 20 questions sont bien détectées
if len(colonnes_q_initial) != 20:
    raise ValueError(
        f"Erreur : {len(colonnes_q_initial)} questions détectées au lieu de 20."
    )

domaines_initial = {
    "DA": colonnes_q_initial[0:5],     # Q1 à Q5
    "BI": colonnes_q_initial[5:10],    # Q6 à Q10
    "DS": colonnes_q_initial[10:15],   # Q11 à Q15
    "IA": colonnes_q_initial[15:20]    # Q16 à Q20
}

for domaine, colonnes in domaines_initial.items():
    print(f"{domaine} : {len(colonnes)} questions")
    for col in colonnes:
        print("  -", col)

DA : 5 questions
  - Q1 - Resume ventes Excel
  - Q2 - Import CSV Excel
  - Q3 - Modele donnees Excel
  - Q4 - Mauvaise pratique visu Excel
  - Q5 - 2e grande valeur Excel
BI : 5 questions
  - Q6 - Mesure vs Colonne DAX
  - Q7 - CA annee precedente DAX
  - Q8 - Vue Modele Power BI
  - Q9 - Acces directeurs regionaux
  - Q10 - 12 commerciaux 3 indicateurs
DS : 5 questions
  - Q11 - Bibliotheque CSV Python
  - Q12 - Overfitting Underfitting
  - Q13 - Segmentation 50000 clients
  - Q14 - Deployer modele Python API
  - Q15 - Valeurs manquantes 30pc
IA : 5 questions
  - Q16 - Role system prompt LLM
  - Q17 - Assistant IA PDF financiers
  - Q18 - Role embedding dans RAG
  - Q19 - Agent IA selection outil
  - Q20 - Sortie fiable LLM tableau


Cellule 7  — Modèle de correction du test initial

In [10]:
# ============================================================
# 2.6. Création du modèle de correction du test initial
# VERSION CORRIGÉE
# ============================================================

# Sécurité : vérifier que les domaines sont bien définis
if "domaines_initial" not in globals():
    raise ValueError("La variable domaines_initial n'existe pas. Exécute d'abord la Cellule 6.")

if len(colonnes_q_initial) != 20:
    raise ValueError(
        f"Erreur : {len(colonnes_q_initial)} questions détectées au lieu de 20."
    )


# Construction du modèle de correction
lignes_correction_initiale = []

for domaine, colonnes in domaines_initial.items():
    for col in colonnes:
        numero = extraire_numero_question_initiale(col)
        
        lignes_correction_initiale.append({
            "numero_question": numero,
            "domaine": domaine,
            "question": col,
            "bonne_reponse": ""
        })


df_correction_initiale_template = pd.DataFrame(lignes_correction_initiale)

# Tri de sécurité Q1 -> Q20
df_correction_initiale_template = df_correction_initiale_template.sort_values(
    "numero_question"
).reset_index(drop=True)


# Export du modèle
PATH_CORRECTION_INITIAL_TEMPLATE = OUTPUT_DIR / "template_correction_test_initial.csv"

df_correction_initiale_template.to_csv(
    PATH_CORRECTION_INITIAL_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)


print("Modèle de correction du test initial généré :")
print(PATH_CORRECTION_INITIAL_TEMPLATE)

print("\nAperçu du modèle de correction :")
display(df_correction_initiale_template)

Modèle de correction du test initial généré :
outputs/02_scoring_pedagogique/template_correction_test_initial.csv

Aperçu du modèle de correction :


,numero_question,domaine,question,bonne_reponse
0,1,DA,Q1 - Resume ventes Excel,
1,2,DA,Q2 - Import CSV Excel,
2,3,DA,Q3 - Modele donnees Excel,
3,4,DA,Q4 - Mauvaise pratique visu Excel,
4,5,DA,Q5 - 2e grande valeur Excel,
5,6,BI,Q6 - Mesure vs Colonne DAX,
6,7,BI,Q7 - CA annee precedente DAX,
7,8,BI,Q8 - Vue Modele Power BI,
8,9,BI,Q9 - Acces directeurs regionaux,
9,10,BI,Q10 - 12 commerciaux 3 indicateurs,


Cellule 8 — Modèles de correction intermédiaire et final

Cette version est meilleure, car elle utilise les fichiers de mapping créés à l’Étape 1
11_mapping_questions_intermediaires.csv
12_mapping_questions_finales.csv

In [11]:
# ============================================================
# 2.7. Création des modèles de correction intermédiaire et final
# VERSION CORRIGÉE AVEC MAPPING DES QUESTIONS
# ============================================================

PATH_MAPPING_MI = INPUT_DIR / "11_mapping_questions_intermediaires.csv"
PATH_MAPPING_FINAL = INPUT_DIR / "12_mapping_questions_finales.csv"

print("Mapping intermédiaire existe :", PATH_MAPPING_MI.exists())
print("Mapping final existe :", PATH_MAPPING_FINAL.exists())


# ============================================================
# 1. Chargement des mappings
# ============================================================

if not PATH_MAPPING_MI.exists():
    raise FileNotFoundError(
        "Le fichier 11_mapping_questions_intermediaires.csv est introuvable. "
        "Vérifie les exports de l'Étape 1."
    )

if not PATH_MAPPING_FINAL.exists():
    raise FileNotFoundError(
        "Le fichier 12_mapping_questions_finales.csv est introuvable. "
        "Vérifie les exports de l'Étape 1."
    )

df_mapping_questions_intermediaires = pd.read_csv(PATH_MAPPING_MI)
df_mapping_questions_finales = pd.read_csv(PATH_MAPPING_FINAL)

print("\nMapping intermédiaire :", df_mapping_questions_intermediaires.shape)
print("Mapping final :", df_mapping_questions_finales.shape)


# ============================================================
# 2. Création du modèle de correction des tests intermédiaires
# ============================================================

df_correction_mi_template = df_mapping_questions_intermediaires.copy()

df_correction_mi_template = df_correction_mi_template.rename(columns={
    "ancienne_colonne": "question_originale",
    "nouvelle_colonne": "question_standardisee"
})

df_correction_mi_template["type_test"] = "intermediaire"
df_correction_mi_template["numero_question"] = (
    df_correction_mi_template["question_standardisee"]
    .str.extract(r"Q(\d+)", expand=False)
    .astype(int)
)

df_correction_mi_template["bonne_reponse"] = ""

df_correction_mi_template = df_correction_mi_template[
    [
        "type_test",
        "parcours",
        "feuille",
        "numero_question",
        "question_standardisee",
        "question_originale",
        "bonne_reponse"
    ]
].sort_values(
    ["parcours", "numero_question"]
).reset_index(drop=True)


# ============================================================
# 3. Création du modèle de correction des tests finaux
# ============================================================

df_correction_final_template = df_mapping_questions_finales.copy()

df_correction_final_template = df_correction_final_template.rename(columns={
    "ancienne_colonne": "question_originale",
    "nouvelle_colonne": "question_standardisee"
})

df_correction_final_template["type_test"] = "final"
df_correction_final_template["numero_question"] = (
    df_correction_final_template["question_standardisee"]
    .str.extract(r"Q(\d+)", expand=False)
    .astype(int)
)

df_correction_final_template["bonne_reponse"] = ""

df_correction_final_template = df_correction_final_template[
    [
        "type_test",
        "parcours",
        "feuille",
        "numero_question",
        "question_standardisee",
        "question_originale",
        "bonne_reponse"
    ]
].sort_values(
    ["parcours", "numero_question"]
).reset_index(drop=True)


# ============================================================
# 4. Export des modèles de correction
# ============================================================

PATH_CORRECTION_MI_TEMPLATE = OUTPUT_DIR / "template_correction_tests_intermediaires.csv"
PATH_CORRECTION_FINAL_TEMPLATE = OUTPUT_DIR / "template_correction_tests_finaux.csv"

df_correction_mi_template.to_csv(
    PATH_CORRECTION_MI_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)

df_correction_final_template.to_csv(
    PATH_CORRECTION_FINAL_TEMPLATE,
    index=False,
    encoding="utf-8-sig"
)


print("\nModèle de correction tests intermédiaires généré :")
print(PATH_CORRECTION_MI_TEMPLATE)

print("\nModèle de correction tests finaux généré :")
print(PATH_CORRECTION_FINAL_TEMPLATE)


print("\nAperçu correction tests intermédiaires :")
display(df_correction_mi_template.head(20))

print("\nAperçu correction tests finaux :")
display(df_correction_final_template.head(20))


print("\nContrôle du nombre de questions par parcours — intermédiaire :")
display(
    df_correction_mi_template
    .groupby("parcours")["question_standardisee"]
    .count()
)

print("\nContrôle du nombre de questions par parcours — final :")
display(
    df_correction_final_template
    .groupby("parcours")["question_standardisee"]
    .count()
)

Mapping intermédiaire existe : True
Mapping final existe : True

Mapping intermédiaire : (60, 4)
Mapping final : (60, 4)

Modèle de correction tests intermédiaires généré :
outputs/02_scoring_pedagogique/template_correction_tests_intermediaires.csv

Modèle de correction tests finaux généré :
outputs/02_scoring_pedagogique/template_correction_tests_finaux.csv

Aperçu correction tests intermédiaires :


,type_test,parcours,feuille,numero_question,question_standardisee,question_originale,bonne_reponse
0,intermediaire,BI,MI-TEST-BI,1,mi_Q01,Quel niveau d'analyse repond a la question : Q...,
1,intermediaire,BI,MI-TEST-BI,2,mi_Q02,Lequel de ces ensembles constitue correctement...,
2,intermediaire,BI,MI-TEST-BI,3,mi_Q03,Quel mode de connexion Power BI est recommande...,
3,intermediaire,BI,MI-TEST-BI,4,mi_Q04,Un analyste Elite Power BI doit principalement...,
4,intermediaire,BI,MI-TEST-BI,5,mi_Q05,"Dans Power Query, l'etape Load du modele ETL c...",
5,intermediaire,BI,MI-TEST-BI,6,mi_Q06,Le Fill Down dans Power Query permet de corrig...,
6,intermediaire,BI,MI-TEST-BI,7,mi_Q07,Quelle est la methode recommandee pour l'Unpiv...,
7,intermediaire,BI,MI-TEST-BI,8,mi_Q08,"Dans l'architecture 3 couches Power Query, que...",
8,intermediaire,BI,MI-TEST-BI,9,mi_Q09,Pourquoi faut-il imperativement typer la colon...,
9,intermediaire,BI,MI-TEST-BI,10,mi_Q10,Quel est l'ordre correct des operations lors d...,



Aperçu correction tests finaux :


,type_test,parcours,feuille,numero_question,question_standardisee,question_originale,bonne_reponse
0,final,BI,FINAL-TEST-BI,1,final_Q01,Une source contient des lignes Total et des en...,
1,final,BI,FINAL-TEST-BI,2,final_Q02,Un champ Matricule contient 00125. Quel type c...,
2,final,BI,FINAL-TEST-BI,3,final_Q03,"Une table a les colonnes Produit, Jan, Fev, Ma...",
3,final,BI,FINAL-TEST-BI,4,final_Q04,Une colonne Code contient MG-TNR-2026. Vous vo...,
4,final,BI,FINAL-TEST-BI,5,final_Q05,"Avant un Merge entre Ventes et Produits, quell...",
5,final,BI,FINAL-TEST-BI,6,final_Q06,Pourquoi construire un modele en etoile au lie...,
6,final,BI,FINAL-TEST-BI,7,final_Q07,"Apres avoir relie Dim_Produit a Fact_Ventes, q...",
7,final,BI,FINAL-TEST-BI,8,final_Q08,Vous devez calculer un taux de marge qui chang...,
8,final,BI,FINAL-TEST-BI,9,final_Q09,Quelle fonction DAX sert a recalculer une mesu...,
9,final,BI,FINAL-TEST-BI,10,final_Q10,"Vous classez les ventes en Faible, Moyen, Fort...",



Contrôle du nombre de questions par parcours — intermédiaire :


parcours
BI    15
DA    15
DS    15
IA    15
Name: question_standardisee, dtype: int64


Contrôle du nombre de questions par parcours — final :


parcours
BI    15
DA    15
DS    15
IA    15
Name: question_standardisee, dtype: int64

Mapping intermédiaire : 60 lignes
Mapping final         : 60 lignes

Intermédiaire :
DA = 15 questions
BI = 15 questions
DS = 15 questions
IA = 15 questions

Final :
DA = 15 questions
BI = 15 questions
DS = 15 questions
IA = 15 questions

Cellule 9 — Chargement et vérification des barèmes complétés

Test initial        : 20 bonnes réponses vides
Tests intermédiaires : 60 bonnes réponses vides
Tests finaux         : 60 bonnes réponses vides

Total : 140 réponses correctes à compléter

Cellule 9 bis — Aide pour voir les réponses disponibles

In [13]:
# ============================================================
# 2.8 bis. Aide au remplissage des barèmes
# Extraction des réponses observées par question
# ============================================================

def extraire_reponses_observees(df, colonnes_questions, type_test, parcours_col=None):
    """
    Extrait les réponses observées pour chaque question.
    Utile pour remplir les barèmes avec les valeurs exactes présentes dans les données.
    """
    lignes = []
    
    if parcours_col is None:
        for question in colonnes_questions:
            valeurs = (
                df[question]
                .dropna()
                .astype(str)
                .str.strip()
            )
            
            comptage = valeurs.value_counts(dropna=False)
            
            for reponse, effectif in comptage.items():
                lignes.append({
                    "type_test": type_test,
                    "parcours": "INITIAL",
                    "question": question,
                    "reponse_observee": reponse,
                    "effectif": int(effectif)
                })
    
    else:
        for parcours in sorted(df[parcours_col].dropna().unique()):
            df_parcours = df[df[parcours_col] == parcours]
            
            for question in colonnes_questions:
                valeurs = (
                    df_parcours[question]
                    .dropna()
                    .astype(str)
                    .str.strip()
                )
                
                comptage = valeurs.value_counts(dropna=False)
                
                for reponse, effectif in comptage.items():
                    lignes.append({
                        "type_test": type_test,
                        "parcours": parcours,
                        "question": question,
                        "reponse_observee": reponse,
                        "effectif": int(effectif)
                    })
    
    return pd.DataFrame(lignes)


# Réponses observées dans le test initial
df_reponses_initial = extraire_reponses_observees(
    df_test_initial,
    colonnes_q_initial,
    type_test="initial",
    parcours_col=None
)

# Réponses observées dans les tests intermédiaires
df_reponses_mi = extraire_reponses_observees(
    df_tests_intermediaires,
    colonnes_q_mi,
    type_test="intermediaire",
    parcours_col="parcours_test_intermediaire"
)

# Réponses observées dans les tests finaux
df_reponses_final = extraire_reponses_observees(
    df_tests_finaux,
    colonnes_q_final,
    type_test="final",
    parcours_col="parcours_test_final"
)


# Export des fichiers d'aide
PATH_REPONSES_INITIAL = OUTPUT_DIR / "aide_reponses_observees_test_initial.csv"
PATH_REPONSES_MI = OUTPUT_DIR / "aide_reponses_observees_tests_intermediaires.csv"
PATH_REPONSES_FINAL = OUTPUT_DIR / "aide_reponses_observees_tests_finaux.csv"

df_reponses_initial.to_csv(PATH_REPONSES_INITIAL, index=False, encoding="utf-8-sig")
df_reponses_mi.to_csv(PATH_REPONSES_MI, index=False, encoding="utf-8-sig")
df_reponses_final.to_csv(PATH_REPONSES_FINAL, index=False, encoding="utf-8-sig")


print("Fichiers d'aide générés :")
print(PATH_REPONSES_INITIAL)
print(PATH_REPONSES_MI)
print(PATH_REPONSES_FINAL)

print("\nAperçu réponses observées — test initial :")
display(df_reponses_initial.head(30))

print("\nAperçu réponses observées — tests intermédiaires :")
display(df_reponses_mi.head(30))

print("\nAperçu réponses observées — tests finaux :")
display(df_reponses_final.head(30))

Fichiers d'aide générés :
outputs/02_scoring_pedagogique/aide_reponses_observees_test_initial.csv
outputs/02_scoring_pedagogique/aide_reponses_observees_tests_intermediaires.csv
outputs/02_scoring_pedagogique/aide_reponses_observees_tests_finaux.csv

Aperçu réponses observées — test initial :


,type_test,parcours,question,reponse_observee,effectif
0,initial,INITIAL,Q1 - Resume ventes Excel,B. Tableau croise dynamique (TCD),110
1,initial,INITIAL,Q1 - Resume ventes Excel,C. Formule SOMME(),67
2,initial,INITIAL,Q1 - Resume ventes Excel,E. Je ne sais pas,55
3,initial,INITIAL,Q1 - Resume ventes Excel,A. Filtre automatique,41
4,initial,INITIAL,Q1 - Resume ventes Excel,D. Graphique en barres,32
5,initial,INITIAL,Q2 - Import CSV Excel,E. Je ne sais pas,130
6,initial,INITIAL,Q2 - Import CSV Excel,D. Power Query,57
7,initial,INITIAL,Q2 - Import CSV Excel,B. Copier-coller manuellement,49
8,initial,INITIAL,Q2 - Import CSV Excel,C. Formules SI imbriquees,42
9,initial,INITIAL,Q2 - Import CSV Excel,A. Macros VBA manuelles,27



Aperçu réponses observées — tests intermédiaires :


,type_test,parcours,question,reponse_observee,effectif
0,intermediaire,BI,mi_Q01,C) Descriptif,41
1,intermediaire,BI,mi_Q01,D) Diagnostique,10
2,intermediaire,BI,mi_Q01,B) Prescriptif,5
3,intermediaire,BI,mi_Q01,A) Predictif,4
4,intermediaire,BI,mi_Q02,"B) Power BI Desktop, Service, Mobile et Data G...",48
5,intermediaire,BI,mi_Q02,A) Power Query uniquement,10
6,intermediaire,BI,mi_Q02,"C) Excel, Access et Tableau",2
7,intermediaire,BI,mi_Q03,C) Import — donnees copiees dans Power BI,45
8,intermediaire,BI,mi_Q03,D) Push Dataset — envoi de donnees en temps reel,8
9,intermediaire,BI,mi_Q03,A) DirectQuery — lecture en direct sur la source,6



Aperçu réponses observées — tests finaux :


,type_test,parcours,question,reponse_observee,effectif
0,final,BI,final_Q01,B) Des mesures fausses car ces lignes peuvent ...,42
1,final,BI,final_Q01,D) Un doublon uniquement visible dans la vue D...,12
2,final,BI,final_Q01,A) Aucun risque si les visuels utilisent des f...,2
3,final,BI,final_Q01,C) Un ralentissement sans impact sur les chiffres,1
4,final,BI,final_Q02,B) Texte,35
5,final,BI,final_Q02,A) Nombre entier avec format 00000,13
6,final,BI,final_Q02,D) Nombre decimal,8
7,final,BI,final_Q02,C) Nombre entier puis conversion dans DAX,3
8,final,BI,final_Q03,"C) Depivoter Jan, Fev, Mar en Mois/Valeur",48
9,final,BI,final_Q03,A) Dupliquer une requete par mois,5


Puis complète les 3 fichiers de correction :
template_correction_test_initial.csv
template_correction_tests_intermediaires.csv
template_correction_tests_finaux.csv

Après avoir rempli les 140 bonnes réponses ,exécute la Cellule 9

Cellule 9 — Chargement et vérification des barèmes complétés

In [ ]:
# ============================================================
# 2.8. Chargement et vérification des barèmes complétés
# ============================================================

PATH_CORRECTION_INITIAL = OUTPUT_DIR / "template_correction_test_initial.csv"
PATH_CORRECTION_MI = OUTPUT_DIR / "template_correction_tests_intermediaires.csv"
PATH_CORRECTION_FINAL = OUTPUT_DIR / "template_correction_tests_finaux.csv"

print("Fichier correction initiale existe :", PATH_CORRECTION_INITIAL.exists())
print("Fichier correction intermédiaire existe :", PATH_CORRECTION_MI.exists())
print("Fichier correction finale existe :", PATH_CORRECTION_FINAL.exists())

if not PATH_CORRECTION_INITIAL.exists():
    raise FileNotFoundError("Le fichier de correction du test initial est introuvable.")

if not PATH_CORRECTION_MI.exists():
    raise FileNotFoundError("Le fichier de correction des tests intermédiaires est introuvable.")

if not PATH_CORRECTION_FINAL.exists():
    raise FileNotFoundError("Le fichier de correction des tests finaux est introuvable.")


# Chargement des fichiers de correction
df_correction_initiale = pd.read_csv(PATH_CORRECTION_INITIAL)
df_correction_mi = pd.read_csv(PATH_CORRECTION_MI)
df_correction_final = pd.read_csv(PATH_CORRECTION_FINAL)


def est_vide(valeur):
    """
    Vérifie si une cellule de bonne réponse est vide.
    """
    if pd.isna(valeur):
        return True
    
    valeur = str(valeur).strip()
    
    return valeur == "" or valeur.lower() in ["nan", "none", "null", "nat"]


def verifier_bareme(df, nom_bareme, nb_lignes_attendu):
    """
    Vérifie qu'un barème est complet :
    - nombre de lignes attendu ;
    - présence de la colonne bonne_reponse ;
    - absence de bonnes réponses vides.
    """
    print(f"\n===== Vérification : {nom_bareme} =====")
    
    print("Dimensions :", df.shape)
    
    if df.shape[0] != nb_lignes_attendu:
        print(f"Attention : {df.shape[0]} lignes trouvées au lieu de {nb_lignes_attendu}.")
    else:
        print("Nombre de lignes : OK")
    
    if "bonne_reponse" not in df.columns:
        raise ValueError(f"Colonne bonne_reponse absente dans {nom_bareme}.")
    
    nb_vides = df["bonne_reponse"].apply(est_vide).sum()
    nb_remplies = df.shape[0] - nb_vides
    
    print("Bonnes réponses remplies :", nb_remplies)
    print("Bonnes réponses vides :", nb_vides)
    
    if nb_vides > 0:
        print("\nLignes encore non complétées :")
        display(df[df["bonne_reponse"].apply(est_vide)].head(20))
    else:
        print("Barème complet : OK")
    
    return nb_vides


nb_vides_initial = verifier_bareme(
    df_correction_initiale,
    "Barème test initial",
    20
)

nb_vides_mi = verifier_bareme(
    df_correction_mi,
    "Barème tests intermédiaires",
    60
)

nb_vides_final = verifier_bareme(
    df_correction_final,
    "Barème tests finaux",
    60
)


total_vides = nb_vides_initial + nb_vides_mi + nb_vides_final

print("\n===== SYNTHÈSE DES BARÈMES =====")
print("Réponses manquantes test initial :", nb_vides_initial)
print("Réponses manquantes tests intermédiaires :", nb_vides_mi)
print("Réponses manquantes tests finaux :", nb_vides_final)
print("Total réponses manquantes :", total_vides)

if total_vides > 0:
    raise ValueError(
        "Les barèmes ne sont pas encore complets. "
        "Complète toutes les colonnes bonne_reponse avant de continuer."
    )
else:
    print("Tous les barèmes sont complets. On peut passer au calcul des scores.")